# 🎙️ Studio Editoriale AI

Questo notebook esegue l'applicazione **Studio Editoriale AI** su Google Colab con accelerazione **GPU T4**, scaricando automaticamente il codice e le voci dal repository GitHub.

### 1. Clonazione del Repository da GitHub
Esegui questa cella per scaricare automaticamente tutti i file del progetto in un click.

In [ ]:
import os

GITHUB_REPO_URL = "https://github.com/emanuelec807/AIStudioEbook.git"
REPO_NAME = "AIStudioEbook"

if not os.path.exists(f"/content/{REPO_NAME}"):
    print(f"⏳ Clonazione del repository {REPO_NAME}...")
    !git clone {GITHUB_REPO_URL} /content/{REPO_NAME}
else:
    print(f"🔄 Aggiornamento repository {REPO_NAME}...")
    %cd /content/{REPO_NAME}
    !git pull

%cd /content/{REPO_NAME}
print(f"\n✅ Repository pronto nella cartella: {os.getcwd()}")

### 2. Installazione delle Dipendenze (in 3 passaggi rapidi)
Esegui questa cella per installare tutte le librerie necessarie (circa 30 secondi in totale).

In [ ]:
# 1. Installa librerie web, audio e Kokoro TTS (compatibile Python 3.13)
print("⏳ [1/3] Installazione Kokoro TTS e librerie applicative...")
!pip install -q soundfile transformers flask flask-cors pydub EbookLib beautifulsoup4 accelerate requests loguru
!pip install -q --no-deps kokoro

# 2. Installa Coqui XTTS con supporto codec audio per PyTorch
print("\n⏳ [2/3] Installazione Coqui XTTS e torchcodec...")
!pip install -q torchcodec "coqui-tts[codec]"

# 3. Installa FFmpeg ed eSpeak per la fonetica audio
print("\n⏳ [3/3] Installazione pacchetti di sistema FFmpeg ed eSpeak-NG...")
!apt-get install -y ffmpeg espeak-ng

print("\n" + "="*50)
print("🎉 TUTTE LE DIPENDENZE INSTALLATE CON SUCCESSO!")
print("👉 Puoi procedere alla Cella 3 (Ollama) oppure direttamente alla Cella 4 per avviare l'app!")
print("="*50)

### 3. (Opzionale) Installazione e Avvio di Ollama per TranslateGemma (con percentuale in tempo reale)
Se desideri utilizzare la traduzione AI con **TranslateGemma 12B e 4B** su Colab, esegui questa cella.

In [ ]:
import subprocess
import time
import requests
import json
import sys
import os

# 1. Installa zstd ed Ollama su Linux
print("⏳ [1/3] Installazione zstd ed Ollama...")
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Avvia il server Ollama in background
print("\n⏳ [2/3] Avvio server Ollama...")
subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=open("ollama.log", "w"))

# Attendi che il server Ollama risponda
print("⏳ Inizializzazione Ollama in corso...")
for _ in range(15):
    try:
        r = requests.get("http://127.0.0.1:11434")
        if r.status_code == 200:
            break
    except Exception:
        pass
    time.sleep(1)

# Funzione per scaricare i modelli con barra di avanzamento e percentuale live
def pull_con_percentuale(model_name):
    print(f"\n⏳ Download di {model_name} in corso...")
    url = "http://127.0.0.1:11434/api/pull"
    try:
        response = requests.post(url, json={"name": model_name, "stream": True}, stream=True)
        last_pct = -1
        for line in response.iter_lines():
            if line:
                data = json.loads(line.decode('utf-8'))
                status = data.get("status", "")
                total = data.get("total", 0)
                completed = data.get("completed", 0)
                if total > 0:
                    pct = int((completed / total) * 100)
                    gb_done = completed / (1024 * 1024 * 1024)
                    gb_tot = total / (1024 * 1024 * 1024)
                    if pct != last_pct:
                        bar = "█" * (pct // 4) + "-" * (25 - (pct // 4))
                        sys.stdout.write(f"\r📥 {model_name}: [{bar}] {pct}% ({gb_done:.2f} GB / {gb_tot:.2f} GB)")
                        sys.stdout.flush()
                        last_pct = pct
                elif status and "pulling" not in status and "downloading" not in status:
                    print(f"\n📦 {status}")
        print(f"\n✅ {model_name} completato con successo!")
    except Exception:
        !ollama pull {model_name}

# 3. Scarica TranslateGemma 12B e 4B mostrando la percentuale
pull_con_percentuale("translategemma:12b")
pull_con_percentuale("translategemma:4b")

print("\n" + "="*50)
print("🎉 OLLAMA & TRANSLATEGEMMA (12B + 4B) PRONTI E ATTIVI!")
print("="*50)

### 4. Avvio dell'Applicazione
Esegui questa cella per avviare il server ed ottenere subito il link di accesso all'applicazione.

In [ ]:
import subprocess
import time
import requests
import re
import os

# 1. Installa tutte le dipendenze audio e server compatibili con Python 3.13
!pip install -q torchcodec "coqui-tts[codec]"
!pip install -q --no-deps kokoro
!pip install -q flask flask-cors pydub EbookLib beautifulsoup4 soundfile accelerate requests loguru

# 2. Termina eventuali istanze server o tunnel precedenti
!pkill -f "server.py" || true
!pkill -f "cloudflared" || true
time.sleep(1)

# 3. Crea cartelle di output se non esistono
os.makedirs("audiolibri_output", exist_ok=True)
os.makedirs("audiolibriEpub", exist_ok=True)

# Scarica file voce_rif_female.wav se non presente
if not os.path.exists("voce_rif_female.wav"):
    print("⏳ Download voce di riferimento preset iniziale...")
    !wget -q https://github.com/DeepMount00/Sibilia-TTS/raw/main/voce_rif_female.wav -O voce_rif_female.wav
    print("✅ Voce preset scaricata!")

# 4. Installa Cloudflare Tunnel se non presente
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("⏳ Download Cloudflare Tunnel...")
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

# 5. Avvia il server Flask in background con licenza Coqui auto-accettata
print("⏳ Avvio del server Python...")
env = os.environ.copy()
env["COQUI_TOS_AGREED"] = "1"
subprocess.Popen(["python", "server.py"], env=env, stdout=open("flask.log", "w"), stderr=open("flask.log", "w"))

# Attendi che il server Flask risponda su 127.0.0.1
server_ok = False
for _ in range(15):
    try:
        r = requests.get("http://127.0.0.1:5000/", timeout=2)
        if r.status_code in [200, 404]:
            server_ok = True
            break
    except Exception:
        pass
    time.sleep(1)

if not server_ok:
    print("\n❌ Errore avvio Flask. Log di diagnostica:")
    if os.path.exists("flask.log"):
        print(open("flask.log").read())
else:
    print("✅ Server Flask pronto ed in ascolto!")
    
    # 6. Avvia Cloudflare Tunnel e mantienilo attivo in continuo
    print("\n⏳ Apertura tunnel Cloudflare...")
    proc = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"], 
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    url_stampato = False
    for line in proc.stdout:
        if not url_stampato:
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                tunnel_url = match.group(0)
                print("\n" + "="*65)
                print("🎉 APPLICAZIONE PRONTA ED ATTIVA!")
                print(f"👉 CLICCA QUI PER APRIRE L'APP:  {tunnel_url}")
                print("="*65 + "\n")
                print("🟢 Il tunnel è attivo in continuo (lascia girare la cella mentre usi l'app).")
                url_stampato = True